# Web Scraper

### Set up Notebook

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install requests beautifulsoup4 tqdm

from tqdm import tqdm               # For progress bar display
import requests                     # To send web requests (download pages + images)
from bs4 import BeautifulSoup       # To parse HTML content from WikiArt pages
import time                         # For rate limiting pauses
import os                           # To build save file paths
import pandas as pd
import csv
import json

### Load in Data

In [ ]:
FILENAME = 'artemis_dataset_release_v0_SyntheticCubismOnly.xlsx'   # CHANGE THIS TO CHANGE WHAT IMAGES ARE BEING SCRAPED
INPUT_FILE_PATH =  f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Excel_File_Cuts/{FILENAME}"

# Load in data file
print("Loading metadata...")
#df = pd.read_excel(INPUT_FILE_PATH, usecols=["painting", "art_style"])
df = pd.read_excel(INPUT_FILE_PATH)

# Make sure needed cols are strings
df["painting"] = df["painting"].astype(str)
df["art_style"] = df["art_style"].astype(str)

# NOT NEEDED ANYMORE: Drop duplicate rows in dataframe
#    Reason: We only want to scrape each painting once
#df = df.drop_duplicates(subset=["painting"])

# Extract unique painting identifiers from dataframe
paintings = df["painting"].unique()

# Count total number of unique paintings after deduplication.
total = len(paintings)
print(f"Total unique paintings: {total}")

Loading metadata...
Total unique paintings: 216


### Define Web Scraper

In [ ]:
# ============================================================
# Web Scraper Config
# ============================================================
# Options: abstract_expressionism, action_painting, analytical_cubism, color_field_painting, cubism, minimalism, synthetic_cubism
ART_STYLE = 'synthetic_cubism'
OUTPUT_ROOT =    f"/content/drive/Shareddrives/LLMs_Art_Project/Data/{ART_STYLE}_images/"       # Path to folder to store scraped images in
LOG_FILE_PATH =  f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Missing_Data_Logs/{ART_STYLE}_missing_images.csv"

#BATCH_SIZE =  500
short_sleep =  0.25  # polite delay
long_sleep  =  4     # after each batch finishes


# Make the output directory
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Track images that failed to download
missing = []

# Extract unique painting identifiers from dataframe
# paintings = df["painting"].unique()

In [ ]:
for idx, painting in enumerate(tqdm(paintings, desc="Downloading images")):
    try:
        # Each painting identifier in dataset looks like "artist_painting-title"
        #     Split it into two parts so we can construct the WikiArt URL
        artist, work = painting.split("_", 1)

        # Build url
        page_url = f"https://www.wikiart.org/en/{artist}/{work}"

        # Request HTML page from WikiArt
        response = requests.get(page_url, headers={'User-Agent': 'Mozilla/5.0'})

        # Parse response using BeautifulSou
        soup = BeautifulSoup(response.text, "html.parser")

        # The main artwork image URL is stored in a <meta> tag with property="og:image"
        image_url = soup.find("meta", property="og:image").get("content")

        # Request actual image file using the extracted image URL
        img_response = requests.get(image_url, headers={'User-Agent': 'Mozilla/5.0'})

        # ---------------- CLEAN FILENAME ----------------
        # Extract the image filename from the URL
        # Example URL: ".../image.jpg?version=123!Large"
        # raw_name = image_url.split("/")[-1].split("?")[0]  # Remove query params
        # base_name = raw_name.split("!")[0]                 # Remove "!Large" or "!HD" suffix

        # Create a clean filename based on dataset naming convention
        filename = f"{painting}.jpeg"
        # ------------------------------------------------

        # Determine the full path where the file should be saved
        save_path = os.path.join(OUTPUT_ROOT, filename)

        # Save the image in binary write mode
        with open(save_path, "wb") as f:
            f.write(img_response.content)

        # Optional progress log
        # tqdm.write(f"Saved: {save_path}")

        # Pause slightly between downloads to avoid server overload
        time.sleep(short_sleep)

        # Take a longer break after every 25 downloads (server courtesy)
        if (idx + 1) % 25 == 0:
            tqdm.write("Batch complete — taking longer break...")
            time.sleep(long_sleep)

    except Exception as e:
        # If something fails, log it
        tqdm.write(f"❌ Failed: {painting}")
        missing.append([painting, str(e)])


# If any downloads failed, save them to a CSV log
if missing:
    pd.DataFrame(missing, columns=["painting", "error"]).to_csv(LOG_FILE_PATH, index=False)
    print("Missing image log saved to:", LOG_FILE_PATH)

print("\nDOWNLOAD COMPLETE")


Batch complete — taking longer break...


Batch complete — taking longer break...


Batch complete — taking longer break...


Batch complete — taking longer break...


Batch complete — taking longer break...


Batch complete — taking longer break...


Batch complete — taking longer break...


Batch complete — taking longer break...



DOWNLOAD COMPLETE
